# Local model training — from scratch
Run cells in order in JupyterLab on your training VM. This notebook calls the existing project trainers rather than duplicating them. No pretrained weights or remote AI services are used.

**Scope:** this trains the existing four-state visual component, not the future language/task-recognition model. GPU access depends on your VM configuration. Training datasets, checkpoints and reports deliberately persist on the training machine; this is separate from active task privacy.

## Hardware target: Arduino UNO Q (4 GB / 32 GB)
Train in the GPU VM. Default deployment is the full INT8 classifier on UNO Q Linux. An optional Linux encoder/STM32 MCU head split needs Bridge integration. This notebook does not flash the board or establish board latency, complete memory use, or mouse/keyboard support.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# Launch Jupyter from the project folder, or set PROJECT_ROOT to its absolute location.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "local_model").is_dir() and (PROJECT_ROOT.parent / "local_model").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "local_model" / "train.py").is_file(), "Set PROJECT_ROOT to the extracted project folder."
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
def run_module(module, *arguments):
    subprocess.run([sys.executable, "-u", "-m", module, *map(str, arguments)], cwd=PROJECT_ROOT, check=True)
print("Project:", PROJECT_ROOT)
print("Kernel Python:", sys.executable)


## Choose a trainer
Use `cuda` for the Ampere VM, `cpu` for PyTorch on CPU, or `numpy` for the verified handwritten CPU trainer. Outputs go into a separate folder so the supplied demonstration model stays available. Install the correct CUDA-enabled PyTorch build for your VM before choosing CUDA.


In [ ]:
BACKEND = "cuda"  # cuda | cpu | numpy
EPOCHS = 60
BATCH_SIZE = 64
SEED = 7
DATASET = PROJECT_ROOT / "data" / "demo_dataset.npz"
OUTPUT = PROJECT_ROOT / "artifacts_jupyter"
assert BACKEND in ("cuda", "cpu", "numpy")
assert EPOCHS > 0 and BATCH_SIZE > 0
if BACKEND != "numpy":
    import torch
    print("PyTorch:", torch.__version__)
    if BACKEND == "cuda":
        assert torch.cuda.is_available(), "CUDA unavailable: check your VM GPU assignment, driver and PyTorch installation."
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory bytes:", torch.cuda.get_device_properties(0).total_memory)
print("Dataset:", DATASET)
print("Output:", OUTPUT)


## Inspect data
The bundled dataset is synthetic. For real data, first create a dataset with the project capture/preparation tools and change `DATASET` above. Keep training, validation and test sessions separate.


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from local_model.train import load_dataset
data = load_dataset(DATASET)
for split in ("train", "val", "test"):
    print(split, data["x_" + split].shape, data["y_" + split].shape)
print(data["metadata"])
del data


## Train from random initialization
Running this cell starts a new training run, not a resume. Re-running it replaces files in the chosen output directory.


In [ ]:
module = "local_model.train" if BACKEND == "numpy" else "local_model.train_gpu"
arguments = ["--data", DATASET, "--outdir", OUTPUT, "--epochs", EPOCHS,
             "--batch-size", BATCH_SIZE, "--seed", SEED]
if BACKEND != "numpy":
    arguments += ["--device", BACKEND]
run_module(module, *arguments)


In [ ]:
report_name = "training_report.json" if BACKEND == "numpy" else "gpu_training_report.json"
report = json.loads((OUTPUT / report_name).read_text())
print(json.dumps({key: value for key, value in report.items() if key != "history"}, indent=2))
history = report.get("history", [])
try:
    import matplotlib.pyplot as plt
    if history:
        plt.plot([row["epoch"] for row in history], [row["val_loss"] for row in history])
        plt.xlabel("Epoch"); plt.ylabel("Validation loss"); plt.show()
except ImportError:
    print("Install matplotlib for optional plots.")


## Export integer model
This produces the project's custom INT8 format and C++ headers, not a LiteRT file. Reported accuracy is only evidence for the supplied dataset.


In [ ]:
run_module("local_model.quantize", "--model", OUTPUT / "model.npz", "--data", DATASET,
           "--outdir", OUTPUT, "--header-dir", OUTPUT / "model_data")
print((OUTPUT / "quantization_report.json").read_text())


## Optional native verification
Requires `g++` on the VM. This checks exported integer outputs against the C++ runtime, not UNO Q deployment.


In [ ]:
VERIFY_CPP = False
if VERIFY_CPP:
    subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "verify_cpp.py"),
                    "--artifact-dir", str(OUTPUT), "--model-data-dir", str(OUTPUT / "model_data"),
                    "--build-dir", str(OUTPUT / "native_build")], cwd=PROJECT_ROOT, check=True)


## Package for UNO Q Linux
Run after export. Transfer the resulting ZIP to the board and follow DEPLOY_UNO_Q.md. This packages OUTPUT, not the supplied demonstration weights.


In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/package_uno_q.py"),
                "--artifact-dir", str(OUTPUT), "--out", str(PROJECT_ROOT / "uno-q-runtime.zip")],
               cwd=PROJECT_ROOT, check=True)
